# Example 12.6.4 / 2: Exact counterpart or finite approximation

Companion notebook for **12.6.4 Robust Counterparts** in the AMPL lab Documentation, section "Exact counterpart or finite approximation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exact counterpart or finite approximation

Checking a finite list of scenarios is exact only when those points characterize the maximum for the set and expression. One example is the set of all relevant vertices of a polytope. Randomly sampled points provide empirical protection, not universal robust feasibility.

On the box example, write one capacity row per listed coefficient point. `POINTS` holds the
nominal point and the four vertices of the box; `USED` selects the rows to impose. The run imposes
the four vertices, then only the nominal point and one vertex, and evaluates each plan's worst case
over the whole box with `get_value`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                                     # decision items
set POINTS;                                    # listed coefficient points
set USED within POINTS;                        # points imposed as rows
param coef {POINTS, ITEMS};                    # coefficient vector of each point
param nominal {ITEMS};                         # nominal coefficient
param deviation {ITEMS} >= 0;                  # box half-width
param value {ITEMS};                           # value per unit of x
param lower {ITEMS};                           # lower bound on x
param upper {ITEMS};                           # upper bound on x
param capacity;                                # right-hand side

var x {i in ITEMS} >= lower[i] <= upper[i];
maximize TotalValue: sum {i in ITEMS} value[i] * x[i];

# One capacity row per imposed coefficient point.
subject to PointCapacity {k in USED}:
    sum {i in ITEMS} coef[k,i] * x[i] <= capacity;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["a", "b"]
ampl.set["POINTS"] = ["nom", "v1", "v2", "v3", "v4"]
ampl.param["coef"] = {
    ("nom", "a"): 2, ("nom", "b"): 1, ("v1", "a"): 1.5, ("v1", "b"): 0.5,
    ("v2", "a"): 1.5, ("v2", "b"): 1.5, ("v3", "a"): 2.5, ("v3", "b"): 0.5,
    ("v4", "a"): 2.5, ("v4", "b"): 1.5,
}
ampl.param["nominal"] = {"a": 2, "b": 1}
ampl.param["deviation"] = {"a": 0.5, "b": 0.5}
ampl.param["value"] = {"a": 3, "b": -1}
ampl.param["lower"] = {"a": 0, "b": -4}
ampl.param["upper"] = {"a": 6, "b": 4}
ampl.param["capacity"] = 10

In [5]:
# run.py
worst = "sum {i in ITEMS} (nominal[i] * x[i] + deviation[i] * abs(x[i]))"
for used in (["v1", "v2", "v3", "v4"], ["nom", "v4"]):
    ampl.set["USED"] = used
    ampl.solve(solver="highs", verbose=False)
    print(used, ampl.var["x"].to_dict(), ampl.obj["TotalValue"].value(),
          ampl.get_value(worst))

['v1', 'v2', 'v3', 'v4'] {'a': 4.8, 'b': -4} 18.4 10
['nom', 'v4'] {'a': 6, 'b': -4} 22.0 13


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := a b;
set POINTS := nom v1 v2 v3 v4;
param coef:  a    b :=
  nom        2    1
  v1       1.5  0.5
  v2       1.5  1.5
  v3       2.5  0.5
  v4       2.5  1.5;
param:  nominal  deviation  value  lower  upper :=
  a           2        0.5      3      0      6
  b           1        0.5     -1     -4      4;
param capacity := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
let USED := {"v1", "v2", "v3", "v4"};
solve;
display x, TotalValue, sum {i in ITEMS} (nominal[i] * x[i] + deviation[i] * abs(x[i]));
let USED := {"nom", "v4"};
solve;
display x, TotalValue, sum {i in ITEMS} (nominal[i] * x[i] + deviation[i] * abs(x[i]));

x [*] :=
a   4.8
b  -4
;

TotalValue = 18.4
sum{i in ITEMS} (nominal[i]*x[i] + deviation[i]*abs(x[i])) = 10

x [*] :=
a   6
b  -4
;

TotalValue = 22
sum{i in ITEMS} (nominal[i]*x[i] + deviation[i]*abs(x[i])) = 13

